# Building an API with Flask and Sakila

**Optional reference and self-study material** — The application is implemented in **`main.py`**, not in this notebook.

In the previous lesson, we used `requests` to **consume** an API. Now we will **build** one that reads data from the Sakila MySQL database.

**Before starting:** follow the setup instructions in the Flask **`README.md`** (Conda, MySQL, `.env`, and running `python main.py`). This notebook explains the main ideas without duplicating the installation instructions.

## 1. From HTTP requests to a Flask application

A browser or Python client sends an HTTP `GET` request. Flask matches the URL to a Python function, which can query MySQL and return a response.

```text
Browser / requests  --GET /actors-->  Flask route  -->  Sakila (MySQL)
Browser / requests  <-- JSON -------- Flask route  <--  rows as dictionaries
```

Our API provides four endpoints:

| Method | Endpoint | Response |
| --- | --- | --- |
| GET | `/` | `Hello world!` |
| GET | `/actors` | Actors from Sakila |
| GET | `/movies` | Movies from Sakila |
| GET | `/search_movies?title=ACADEMY` | Movies whose titles contain `ACADEMY` |

The Flask development server runs at `http://127.0.0.1:5000`.

## 2. Create the application and connect to Sakila

The beginning of **`main.py`** imports the libraries, loads credentials from `.env`, and creates the Flask application:

```python
import os
import pymysql
from dotenv import load_dotenv
from flask import Flask, jsonify, request

load_dotenv()
app = Flask(__name__)
```

The connection helper keeps the MySQL configuration in one place:

```python
def get_db_connection():
    return pymysql.connect(
        host="localhost",
        user=os.getenv("db_username"),
        password=os.getenv("db_password"),
        database="sakila",
        cursorclass=pymysql.cursors.DictCursor,
    )
```

- `load_dotenv()` makes the variables from `.env` available through `os.getenv()`.
- `DictCursor` returns database rows as **dictionaries**, rather than tuples.
- Never put passwords directly in your source code or commit `.env` to Git.

**Important:** This notebook is not the application. Keep these definitions in `main.py`.

## 3. Define routes with decorators

The simplest endpoint does not need a database connection:

```python
@app.route("/", methods=["GET"])
def hello():
    return "Hello world!"
```

The `@app.route(...)` decorator associates a URL and HTTP method with a Python function.

Try `http://127.0.0.1:5000/` in your browser **after starting** the server with `python main.py`.

## 4. Read data from MySQL and return JSON

The `/actors` route illustrates the database workflow used by both `/actors` and `/movies`:

```python
@app.route("/actors", methods=["GET"])
def get_actors():
    try:
        connection = get_db_connection()
        try:
            with connection.cursor() as cursor:
                cursor.execute(
                    "SELECT actor_id, first_name, last_name FROM actor;"
                )
                actors = cursor.fetchall()
        finally:
            connection.close()
    except pymysql.MySQLError:
        app.logger.exception("Database error in /actors")
        return jsonify({"error": "Database request failed."}), 500

    return jsonify(actors)
```

Follow the sequence:

1. Open a MySQL connection.
2. Execute the SQL query and retrieve the rows with `fetchall()`.
3. Close the connection in `finally`, even if the query fails.
4. Convert the rows to a JSON response with `jsonify()`.

If MySQL raises a `pymysql.MySQLError`, the server **logs** the technical details and sends a generic HTTP **500** response to the client. The `/movies` endpoint uses the same pattern with the `film` table.

**Why not `print()`?** `app.logger.exception(...)` records an error message **and the exception traceback** in the server logs. This makes debugging easier without exposing internal details to API users.

## 5. Search movies with a query parameter

Consider the URL:

`http://127.0.0.1:5000/search_movies?title=ACADEMY`

- `/search_movies` is the **path**.
- `title=ACADEMY` is a **query parameter**, introduced by `?`.

The beginning of the endpoint reads and validates the parameter:

```python
title = request.args.get("title", "").strip()

if not title:
    return jsonify({"error": "Title parameter is required."}), 400
```

In `main.py`, the search uses a parameterized SQL query:

```python
safe_title = title.replace("!", "!!").replace("%", "!%").replace("_", "!_")
pattern = f"%{safe_title}%"

query = """
    SELECT film_id, title, description, release_year
    FROM film
    WHERE title LIKE %s ESCAPE '!';
"""
cursor.execute(query, (pattern,))
movies = cursor.fetchall()
```

The `%` characters around the search term mean **contains**. SQL `LIKE` performs partial matching, not fuzzy matching. The `cursor.execute(..., (pattern,))` placeholder keeps input separate from SQL syntax. The `safe_title` step also treats user-supplied `%`, `_`, and `!` as literal characters rather than search operators.

Expected outcomes:

| Request | Result |
| --- | --- |
| `/search_movies?title=ACADEMY` | Matching movies, HTTP 200 |
| `/search_movies` | Missing parameter, HTTP 400 |
| `/search_movies?title=` | Empty parameter, HTTP 400 |
| `/search_movies?title=XYZNOTFOUND` | Empty list `[]`, HTTP 200 |

The complete route, including connection handling and logging, is in **`main.py`**.

## 6. Run and test the application

Open a terminal in the folder containing `main.py`, activate your Conda environment as described in the README, and run:

```bash
python main.py
```

The server continues running in that terminal while it waits for requests. Use your browser or a **second terminal** to test it. Press **Ctrl+C** in the server terminal to stop it.

`debug=True` in `main.py` enables automatic reloading when you save the file. **Do not use debug mode in production.**

### Consume the API with `requests`

Once Flask is running, the following cell can be executed **from this notebook**. It sends a request to the application in `main.py`.

In [ ]:
import requests
import pandas as pd

response = requests.get(
    "http://127.0.0.1:5000/search_movies",
    params={"title": "ACADEMY"},
    timeout=10,
)
response.raise_for_status()

movies = response.json()
pd.DataFrame(movies).head()

The client receives JSON over HTTP; it does **not** need direct access to MySQL. If this cell raises a connection error, first check that `python main.py` is running in another terminal.

## 7. Quick checks (optional)

1. Why does `/actors` use `jsonify()` while `/` can return a simple string?
2. Why is `connection.close()` inside `finally`?
3. What is the difference between HTTP **400** and HTTP **500** in our application?
4. What changes in the request when you replace `params={"title": "ACADEMY"}` with `params={"title": "LOVE"}`?

## 8. Optional practice

**Exercise A — Test the existing API:** With the server running, request `/actors` using `requests`, convert the JSON response to a DataFrame, and display the first five rows. No changes to `main.py` are needed.

In [ ]:
# Optional exercise A: complete the code.
# response = requests.get("...", timeout=10)
# response.raise_for_status()
# actors = response.json()
# pd.DataFrame(actors).head()

**Exercise B — Extend the API (optional challenge):** In a copy of `main.py`, add `GET /actor_count` returning the number of actors in Sakila as JSON. Reuse the same connection, `try`/`finally`, and logging pattern. Test it in your browser.

Suggested SQL: `SELECT COUNT(*) AS actor_count FROM actor;`

Keep this exercise outside the notebook: the Flask server belongs in `main.py`.

## 9. Takeaways and next step

- Flask maps HTTP requests to Python functions using route decorators.
- `pymysql` connects the application to Sakila; `DictCursor` provides dictionary rows.
- `jsonify()` produces JSON responses for the client.
- Query parameters let clients filter results; validate inputs and use parameterized SQL.
- `try`/`finally` closes database connections; logging helps diagnose failures.

**Next:** Build the **same API** with FastAPI and compare the route declarations, JSON responses, error handling, and automatically generated documentation.

For installation, credentials, endpoint URLs, and troubleshooting, consult the Flask **`README.md`**. For the complete implementation, consult **`main.py`**.